# End-to-End Embedding (Bi-Encoder) Fine-Tuning with NeMo AutoModel

A step-by-step tutorial for training production text embedding models using [NeMo AutoModel](https://github.com/NVIDIA-NeMo/Automodel).

We will adapt **Llama-3.2-1B** into a high-performance **dense embedding bi-encoder** for semantic search, document retrieval, and retrieval-augmented generation (RAG).

## Learning Goals & Architectural Concepts

Text embedding models map variable-length text into fixed-dimensional dense vectors where semantic similarity corresponds to geometric proximity (such as high cosine similarity).

### 1. Bi-Encoders vs. Cross-Encoders

| Property | Bi-Encoder (Embedding Model) | Cross-Encoder (Reranker) |
| :--- | :--- | :--- |
| **Architecture** | Encodes query and passage independently: $\mathbf{u} = f(q), \mathbf{v} = g(p)$ | Encodes query and passage jointly: $s = f(q \circ p)$ |
| **Query-Doc Interaction** | Late interaction via dot product / cosine similarity | Full cross-attention across all token positions |
| **Indexing & Scalability** | Pre-compute and index all corpus embeddings offline (ANN search over millions of docs in <5 ms) | Must recompute forward pass for every candidate at query time |
| **Accuracy** | High (ideal for first-stage retrieval) | Very High (ideal for second-stage reranking over top-100 candidates) |

### 2. Contrastive Learning & the InfoNCE Objective

Bi-encoders are trained using the **InfoNCE contrastive loss**. Given a query $q_i$, its positive passage $p_i^+$, and a set of negative passages $\{p_{i,j}^-\}$, the loss maximizes the agreement between the positive pair relative to all negatives:

$$\mathcal{L}_i = -\log \frac{\exp(\text{sim}(\mathbf{q}_i, \mathbf{p}_i^+) / \tau)}{\exp(\text{sim}(\mathbf{q}_i, \mathbf{p}_i^+) / \tau) + \sum_{j} \exp(\text{sim}(\mathbf{q}_i, \mathbf{p}_{i,j}^-) / \tau)}$$

- **Temperature $\tau$**: Scaling factor (typically $0.02$). Lower temperature sharpens the distribution, forcing the model to push hard negatives further away in embedding space.
- **In-Batch Negatives**: Passages from other queries in the same mini-batch are treated as negatives for query $i$. In a batch of size $B$, each query gets $B-1$ free negative examples.
- **Hard Negatives**: Specially mined non-relevant passages that share surface or lexical keywords with the query. Mined hard negatives force the model to learn deep conceptual semantics rather than relying on trivial lexical mismatches.

### 3. Bidirectional Attention for Generative Backbones

Standard autoregressive models (like Llama) use **causal attention masks** to prevent tokens from attending to future positions. However, for embedding representation, every token must be able to attend to the entire context.

`NeMoAutoModelBiEncoder` automatically converts standard causal Hugging Face backbones into **bidirectional models** (`is_causal: false`), producing richer, holistic sentence representations.

### 4. Pooling Strategies & Normalization

- **Mean (Average) Pooling**: Averages token embeddings across the sequence length, weighted by the attention mask. This is the standard choice for embedding backbones.
- **L2 Normalization**: Every output embedding $\mathbf{v}$ is normalized such that $\|\mathbf{v}\|_2 = 1.0$. This ensures cosine similarity simplifies directly to an inner dot product:

$$\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2} = \mathbf{u} \cdot \mathbf{v}$$

---
## Prerequisites & Setup

* **Hardware**: 1 or more NVIDIA GPUs with at least 16 GB VRAM (e.g. L40S, A100, RTX 4090).
* **Software**: NeMo AutoModel installed (`pip install -e .` or container environment).
* **Hugging Face Authentication**: Access to Llama-3.2-1B (`huggingface-cli login` or export `HF_TOKEN`).

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

---
## Step 1. Prepare Retrieval Training Data

AutoModel supports JSONL retrieval datasets where each line contains a JSON object with:
- `question`: The input query string.
- `pos_doc`: A list containing the ground-truth relevant document(s).
- `neg_doc`: A list containing one or more negative / non-relevant documents.

Let us inspect the bundled sample retrieval dataset:

In [ ]:
import json
from pathlib import Path

data_path = Path("sample_retrieval_data.jsonl")
with open(data_path, "r", encoding="utf-8") as f:
    samples = [json.loads(line) for line in f]

print(f"Loaded {len(samples)} sample retrieval pairs.\n")
first = samples[0]
print("Query:", first["question"])
print("Positive Passage:", first["pos_doc"][0][:120], "...")
print("Negative Passage:", first["neg_doc"][0][:120], "...")

### Hard Negative Mining

To train competitive retrieval models, datasets must include **mined hard negatives**. AutoModel provides an automated mining tool at `examples/retrieval/data_utils/mine_hard_negatives.py`:

```bash
python examples/retrieval/data_utils/mine_hard_negatives.py \
    --config examples/retrieval/data_utils/mining_config.yaml \
    --mining.model_name_or_path meta-llama/Llama-3.2-1B \
    --mining.train_qa_file_path raw_qa_pairs.json \
    --mining.train_file_output_path mined_train_data.json \
    --mining.hard_neg_margin 0.95
```

This indexes the corpus, retrieves top candidates for each query using an initial encoder, and filters out false negatives using the `hard_neg_margin` threshold.

---
## Step 2. Configure the Bi-Encoder Training Recipe

AutoModel uses declarative YAML recipes to configure the model, dataset, optimizer, and training schedule.
Let us inspect the configuration in `llama_embed_config.yaml`:

In [ ]:
with open("llama_embed_config.yaml", "r", encoding="utf-8") as f:
    print(f.read())

### Key Configuration Parameters Explained

1. **`recipe: TrainBiEncoderRecipe`**: Selects AutoModel's dedicated contrastive bi-encoder training pipeline with InfoNCE loss and in-batch negative support.
2. **`temperature: 0.02`**: Sharpens cosine similarities. Without this scaling, logits stay within $[-1, 1]$, resulting in diffuse probability distributions and weak gradient signals.
3. **`model._target_: nemo_automodel.NeMoAutoModelBiEncoder.from_pretrained`**: The AutoModel entry point that automatically removes causal masks, wraps the backbone, and applies average pooling and L2 normalization.
4. **`query_prefix: "query: "` / `passage_prefix: "passage: "`**: Asymmetric task prefixes instruct the encoder whether it is embedding an asymmetric search query or an indexable passage.
5. **`save_consolidated: true`**: Automatically exports a single, unified Hugging Face compatible `model.safetensors` file at checkpoint time.

---
## Step 3. Launching Bi-Encoder Fine-Tuning

You can launch the training job directly using the AutoModel CLI or programmatically in Python.

### CLI Launch Command

```bash
# Single-GPU training:
automodel tutorials/embedding-biencoder/llama_embed_config.yaml --nproc-per-node 1

# Multi-GPU training with FSDP2:
automodel tutorials/embedding-biencoder/llama_embed_config.yaml --nproc-per-node 8
```

### Programmatic Execution in Python

In [ ]:
from nemo_automodel.components.config._arg_parser import parse_args_and_load_config

# Load recipe config
cfg = parse_args_and_load_config("llama_embed_config.yaml")
print(f"Configured recipe: {cfg.recipe}")
print(f"Model backbone:    {cfg.model.pretrained_model_name_or_path}")
print(f"Temperature:       {cfg.temperature}")
print(f"Pooling:           {cfg.model.pooling}")
print(f"L2 Normalization:  {cfg.model.l2_normalize}")

# To execute training, uncomment:
# recipe = TrainBiEncoderRecipe(cfg)
# recipe.setup()
# recipe.run()

### Expected Training Dynamics

During training, the loss will decrease steadily as positive similarities approach $1.0$ and negative similarities drop toward $0.0$ or below:

```
Step 10/100  | loss: 2.761 | lr: 5.0e-06 | grad_norm: 0.84
Step 25/100  | loss: 1.423 | lr: 4.2e-06 | grad_norm: 0.61
Step 50/100  | loss: 0.782 | lr: 2.8e-06 | grad_norm: 0.45
Step 100/100 | loss: 0.315 | lr: 5.0e-07 | grad_norm: 0.28
```

---
## Step 4. Inference, Semantic Search, & Verification

Once training completes, we can load the bi-encoder and verify that query embeddings correctly match their corresponding positive documents.

In [ ]:
import torch
from transformers import AutoTokenizer

from nemo_automodel import NeMoAutoModelBiEncoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "meta-llama/Llama-3.2-1B"

# Load tokenizer and bi-encoder
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Demonstrate model loading with bidirectional conversion
print(f"Loading {model_name} as NeMoAutoModelBiEncoder (pooling=avg, l2_norm=True)...")
model = NeMoAutoModelBiEncoder.from_pretrained(
    model_name,
    pooling="avg",
    l2_normalize=True,
    is_causal=False,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
).to(device)
model.eval()
print("Bi-Encoder successfully loaded!")

### Computing Similarity Across Queries and Passages

In [ ]:
queries = [
    "query: How does contrastive loss work?",
    "query: What is the benefit of bi-encoders for search?",
]

passages = [
    "passage: Contrastive InfoNCE loss pulls representations of semantically related pairs closer while pushing negatives away.",
    "passage: Bi-encoders compute independent dense vectors that can be pre-indexed for fast sub-millisecond approximate nearest neighbor search.",
    "passage: Stochastic gradient descent computes gradients over miniature random subsets of the training set.",
]

# Tokenize inputs
q_inputs = tokenizer(queries, padding=True, truncation=True, return_tensors="pt").to(device)
p_inputs = tokenizer(passages, padding=True, truncation=True, return_tensors="pt").to(device)

# Generate normalized embeddings
with torch.no_grad():
    q_embeds = model(**q_inputs).sentence_embedding  # [N_queries, D]
    p_embeds = model(**p_inputs).sentence_embedding  # [N_passages, D]

# Compute cosine similarity matrix (inner dot product of unit-length vectors)
sim_matrix = torch.matmul(q_embeds, p_embeds.T)

print("Cosine Similarity Matrix:")
for i, q in enumerate(queries):
    print(f"\nQuery {i + 1}: {q}")
    best_idx = torch.argmax(sim_matrix[i]).item()
    for j, p in enumerate(passages):
        marker = " <-- [Top Match]" if j == best_idx else ""
        print(f"  Passage {j + 1} score: {sim_matrix[i, j]:.4f}{marker}")

---
## Step 5. Export & Production Deployment

Because `save_consolidated: true` was enabled in `llama_embed_config.yaml`, the checkpointer exports standard Hugging Face compatible artifacts into `./output/llama_embed_tutorial/checkpoints/model/consolidated/`:

- `model.safetensors`
- `config.json`
- `tokenizer.json` / `tokenizer_config.json`

You can load this exported checkpoint in downstream RAG frameworks (LangChain, LlamaIndex, vLLM, or sentence-transformers) directly:

```python
from sentence_transformers import SentenceTransformer

# Load trained model directly
embedder = SentenceTransformer("./output/llama_embed_tutorial/checkpoints/model/consolidated")
embeddings = embedder.encode(["Hello world", "Semantic search with NeMo AutoModel"])
print("Embeddings shape:", embeddings.shape)
```

## Summary & Next Steps

In this tutorial, you learned:
1. How bi-encoders decouple query and passage encoding for scalable semantic search.
2. The mathematics behind contrastive InfoNCE loss, temperature scaling, and in-batch / hard negatives.
3. How `NeMoAutoModelBiEncoder` converts causal language models into bidirectional encoders with mean pooling.
4. How to configure and launch training with `TrainBiEncoderRecipe`.
5. How to verify cosine similarity rankings and export consolidated checkpoints for production RAG pipelines.

For reranking top-k candidates after initial retrieval, see the [Reranking Models](/model-coverage/reranking-models/overview) guide.